In [1]:
import os, time, math
from collections import Counter
import numpy as np
import pandas as pd
from datasets import load_dataset
from openai import OpenAI
from tqdm.auto import tqdm

In [ ]:
# CONFIG for Local LM Studio over LAN
LOCAL_BASE_URL = "http://172.17.103.35:1234/v1"
N_QUESTIONS = 400  
OUTPUT_DIR = "results_final_offline"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize the OpenAI client pointing to the networked LM Studio server
client = OpenAI(base_url=LOCAL_BASE_URL, api_key="lm-studio")

# Automatically fetch the loaded model from LM Studio
try:
    available_models = client.models.list()
    MODEL_ID = available_models.data[0].id
    print(f"✅ Auto-detected networked model: {MODEL_ID}")
except Exception as e:
    print(f"❌ Could not connect to LM Studio at {LOCAL_BASE_URL}.")
    print("Ensure the server is running and configured to accept external connections.")
    MODEL_ID = "local-model"

✅ Auto-detected networked model: llama3-med42-8b


In [6]:
# Load MedMCQA with options
ds = load_dataset("medmcqa", "default", split=f"validation[:{N_QUESTIONS}]")
df = pd.DataFrame(ds)

# Options A-D and correct letter
df['A'] = df['opa']
df['B'] = df['opb']
df['C'] = df['opc']
df['D'] = df['opd']
df['correct_letter'] = df['cop'].map({0:'A', 1:'B', 2:'C', 3:'D'})

def sample_answers(prompt, m=6, temp=0.8):
    """Sample m answers from the local model."""
    answers = []
    for _ in tqdm(range(m), desc="Sampling answers", leave=False):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=[{"role": "user", "content": prompt}],
                temperature=temp,
                max_tokens=64
            )
            
            text = resp.choices[0].message.content
            if text:
                answers.append(text.strip())
            else:
                print("⚠️ Warning: Empty response.")
                answers.append("LETTER: X\nREASON: Response empty.")
                
        except Exception as e:
            print(f"❌ API Error: {e}")
            answers.append("LETTER: X\nREASON: API Error occurred.")
            
    return answers

def normalize_answer(question, options, raw_answer):
    """Map raw answer -> LETTER + canonical reason using the local model."""
    A, B, C, D = options
    norm_prompt = f"""Medical evaluator: map answer to exact option letter + short standardized reason.

Question: {question}

Options:
A. {A}
B. {B}
C. {C}
D. {D}

Model answer: {raw_answer}

Respond exactly:
LETTER: A/B/C/D/X
REASON: [1 sentence explanation]"""

    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": norm_prompt}],
            temperature=0.0, # Greedy decoding for normalization
            max_tokens=80
        )
        text = resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ API Error during normalization: {e}")
        text = "LETTER: X\nREASON: Normalization failed."
    
    letter = "X"
    reason = text
    for line in text.splitlines():
        if line.upper().startswith("LETTER:"):
            # Extract just the letter
            for c in line.upper().replace("LETTER:", "").strip():
                if c in "ABCDX":
                    letter = c
                    break
        if line.upper().startswith("REASON:"):
            reason = line[7:].strip()
    return letter, reason

def semantic_entropy(reasons):
    """Entropy over canonical reasons (same reason = same semantic cluster)."""
    counts = Counter(reasons)
    total = sum(counts.values())
    probs = [c/total for c in counts.values()]
    return -sum(p*math.log(p+1e-12) for p in probs)

print(f"✅ Loaded {len(df)} questions ready for evaluation.")

✅ Loaded 400 questions ready for evaluation.


In [ ]:
# --- MAIN LOOP ---
records = []
csv_path = f"{OUTPUT_DIR}/LLAMA_MED42_offline.csv"

# Outer tqdm loop for overall dataset progress
for i in tqdm(range(len(df)), desc="Overall Progress"):
    row = df.iloc[i]
    q = row['question']
    options = (row['A'], row['B'], row['C'], row['D'])
    correct = row['correct_letter']
    
    prompt = f"""Medical MCQ. Choose A/B/C/D and explain briefly.

Question: {q}

A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Answer:"""

    raw_answers = sample_answers(prompt, m=6, temp=0.8)
    
    norm_letters = []
    norm_reasons = []
    
    # Inner tqdm loop for normalization progress
    for ans in tqdm(raw_answers, desc="Normalizing", leave=False):
        letter, reason = normalize_answer(q, options, ans)
        norm_letters.append(letter)
        norm_reasons.append(reason)
    
    H = semantic_entropy(norm_reasons)
    
    letter_counts = Counter(norm_letters)
    maj_letter, maj_count = letter_counts.most_common(1)[0]
    
    maj_reasons = [r for l, r in zip(norm_letters, norm_reasons) if l == maj_letter]
    maj_reason = Counter(maj_reasons).most_common(1)[0][0]
    
    is_correct = (maj_letter == correct)
    hallucinated = (maj_letter != "X" and maj_letter != correct)
    
    records.append({
        'question_id': i,
        'question': q,
        'option_A': options[0],
        'option_B': options[1],
        'option_C': options[2],
        'option_D': options[3],
        'correct_letter': correct,
        'majority_letter': maj_letter,
        'majority_reason': maj_reason,
        'semantic_entropy': H,
        'is_correct': is_correct,
        'hallucinated': hallucinated,
        'samples_count': len(raw_answers),
    })
    
    # --- Real-Time Metric Calculation ---
    acc = np.mean([r['is_correct'] for r in records])
    halluc_rate = np.mean([r['hallucinated'] for r in records])
    
    print(f"\n--- Progress: {i+1}/{len(df)} ---")
    print(f"Real-time Acc: {acc:.1%} | Real-time Hallucination: {halluc_rate:.1%}")
    
    # --- Save Every 2 Questions ---
    if (i + 1) % 2 == 0:
        pd.DataFrame(records).to_csv(csv_path, index=False)
        print(f"💾 Saved intermediate results to {csv_path}")
    
    # Local models don't have rate limits, but a small pause helps manage queueing
    time.sleep(0.1) 

# --- FINAL RESULTS ---
res_df = pd.DataFrame(records)
total = len(res_df)
correct_count = res_df['is_correct'].sum()
halluc = res_df['hallucinated'].sum()

print("\n🎯 MEDICAL QA HALLUCINATION BENCHMARK (OFFLINE)")
print("=" * 50)
print(f"Model: {MODEL_ID}")
print(f"Accuracy: {(correct_count/total)*100:.1f}% ({correct_count}/{total})")
print(f"Hallucination rate: {(halluc/total)*100:.1f}% ({halluc}/{total})")

res_df.to_csv(csv_path, index=False)
print("💾 Final Save:", csv_path)

Overall Progress:   0%|          | 0/400 [00:00<?, ?it/s]

Sampling answers:   0%|          | 0/6 [00:00<?, ?it/s]

Normalizing:   0%|          | 0/6 [00:00<?, ?it/s]


--- Progress: 1/400 ---
Real-time Acc: 100.0% | Real-time Hallucination: 0.0%


Sampling answers:   0%|          | 0/6 [00:00<?, ?it/s]

...

Sampling answers:   0%|          | 0/6 [00:00<?, ?it/s]

Normalizing:   0%|          | 0/6 [00:00<?, ?it/s]


--- Progress: 69/400 ---
Real-time Acc: 55.1% | Real-time Hallucination: 40.6%


Sampling answers:   0%|          | 0/6 [00:00<?, ?it/s]

Normalizing:   0%|          | 0/6 [00:00<?, ?it/s]